## Clase 7 - Ejercicio: algoritmo de búsqueda para la Torre de Hanoi

LEANDRO FABRIZIO BIONDI

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None` si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [1]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi
from aima_libs import aima

In [2]:
def A_estrella_algorithm(number_disks=5) -> tuple[NodeHanoi, dict]:

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)


    # Inicializamos las salidas, pero reemplazar con lo que se quiera usar.
    metrics = {
        "solution_found": True,
        "nodes_explored": None,
        "states_visited": None,
        "nodes_in_frontier": None,
        "max_depth": None,
        "cost_total": None,
    }
    # Se crea el nodo inicial a partir del estado inicial y se guarda en solution
    solution = NodeHanoi(initial_state)
    initial_node = solution

    ## Funcion para comparar el estado actual con el estado final y contar cuantos discos estan en la posicion correcta.
    ## Compara los discos de la torre destino del estado actual con los discos de la torre del estado final.
    ## Devuelve la cantidad de discos que estan en la posicion correcta.
    def discos_correctos(state, goal_state):
            
            idx_destino = None
            for i, rod in enumerate(goal_state.rods):
                if rod:
                    idx_destino = i
                    break
            torre_actual = state.rods[idx_destino]
            torre_meta = goal_state.rods[idx_destino]
            k = 0
            for a, b in zip(torre_actual, torre_meta):
                if a != b:
                    break
                k += 1
            return k
    
    ## Funcion heuristica que devuelve la cantidad de discos que no estan en la posicion correcta.
    def h_base(state):
        k = discos_correctos(state, goal_state)
        return state.number_of_disks - k
    
    ## Funcion heuristica mejorada que devuelve la cantidad de discos que no estan en la posicion correcta
    ## sumandole 1 si la torre tiene algun disco que no esta en la posicion correcta considerando que va a haber un costo
    ## adicional por acomodarlo.
    ## Ejemplo: Si la torre tiene el disco 5 y 4 apilados y sobre ellos esta el disco 1, aun si la torre
    ## tiene la secuencia 5 y 4 correcta, para seguir acomodando los demas discos primero tiene que quitar el disco 1
    ## de arriba. 
    def h_mejorada(state):
        k = discos_correctos(state, goal_state)
        faltantes = state.number_of_disks - k
        idx_destino = None
        for i, rod in enumerate(goal_state.rods):
            if rod:
                idx_destino = i
                break
        torre_actual = state.rods[idx_destino]
        extra = 1 if len(torre_actual) > k else 0
        return faltantes + extra

    # Se crea la frontera como una cola de prioridad donde se ordean de menor a mayor segun
    # el costo del camino recorrido mas la heuristica del estado del nodo.
    # Para usar la heuristica base, reemplazar h_mejorada por h_base y viseversa.
    frontier = aima.PriorityQueue(order = 'min', f=lambda node: node.path_cost + h_mejorada(node.state))
    # Se agrega el nodo inicial a la frontera,
    frontier.append(initial_node)
    # Se inicializa el conjunto de estados explorados, un contador de nodos explorados y la profundidad maxima alcanzada.
    explored = set()
    nodes_explored = 0
    max_depth = 0

    ## Se realiza la busqueda hasta que se encuentre la solucion o se agoten los nodos en la frontera.
    while len(frontier) > 0:
        # Toma el primer nodo de la cola de la frontera
        _, node = frontier.pop()
        # Si el estado del nodo ya fue explorado, lo saltea y continua.
        if node.state in explored:
            continue
        # Registra el nodo como explorado y actualiza el contador.
        explored.add(node.state)
        nodes_explored += 1
        # Actualiza la profundidad maxima alcanzada si es que es mayor que la actual.
        max_depth = max(max_depth, node.depth)
        # Si el estado del nodo es el estado final, se retorna el nodo y las metricas.
        if problem.goal_test(node.state):
            metrics = {
                "solution_found": True,
                "nodes_explored": nodes_explored,
                "states_visited": len(explored),
                "nodes_in_frontier": len(frontier),
                "max_depth": max_depth,
                "cost_total": node.path_cost,
            }
            return node, metrics
        # Si no es el estado final, se expande el nodo creando sus hijos y estos se agregan a la frontera.
        for child in node.expand(problem):
            if child.state not in explored:
                frontier.append(child)
    # Si se agotan los nodos en la frontera y no se encuentra la solucion, retorna Solution = none y las metricas.
    metrics = {
        "solution_found": False,
        "nodes_explored": nodes_explored,
        "states_visited": len(explored),
        "nodes_in_frontier": len(frontier),
        "max_depth": max_depth,
        "cost_total": None,
    }
    return None, metrics


Se prueba la función:

In [3]:
solution, metrics = A_estrella_algorithm(number_disks=5)

Veamos las métricas:

In [4]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 159
states_visited: 159
nodes_in_frontier: 16
max_depth: 31
cost_total: 31.0


Veamos el camino de estados desde el principio a la solución:

In [5]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 | 2 1 | >
<Node HanoiState: 5 4 | 2 1 | 3>
<Node HanoiState: 5 4 1 | 2 | 3>
<Node HanoiState: 5 4 1 |  | 3 2>
<Node HanoiState: 5 4 |  | 3 2 1>
<Node HanoiState: 5 | 4 | 3 2 1>
<Node HanoiState: 5 | 4 1 | 3 2>
<Node HanoiState: 5 2 | 4 1 | 3>
<Node HanoiState: 5 2 1 | 4 | 3>
<Node HanoiState: 5 2 1 | 4 3 | >
<Node HanoiState: 5 2 | 4 3 | 1>
<Node HanoiState: 5 | 4 3 2 | 1>
<Node HanoiState: 5 | 4 3 2 1 | >
<Node HanoiState:  | 4 3 2 1 | 5>
<Node HanoiState: 1 | 4 3 2 | 5>
<Node HanoiState: 1 | 4 3 | 5 2>
<Node HanoiState:  | 4 3 | 5 2 1>
<Node HanoiState: 3 | 4 | 5 2 1>
<Node HanoiState: 3 | 4 1 | 5 2>
<Node HanoiState: 3 2 | 4 1 | 5>
<Node HanoiState: 3 2 1 | 4 | 5>
<Node HanoiState: 3 2 1 |  | 5 4>
<Node HanoiState: 3 2 |  | 5 4 1>
<Node HanoiState: 3 | 2 | 5 4 1>
<Node HanoiState: 3 | 2 1 | 5 4>
<Node HanoiState:  | 2 1 | 5 4 3>
<Node HanoiState: 1 | 2 | 5 4 

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [6]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 4 from 1 to 2
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 3 from 3 to 2
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 5 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
Move disk 3 from 2 to 1
Move disk 1 from 3 to 2
Move disk 2 from 3 to 1
Move disk 1 from 2 to 1
Move disk 4 from 2 to 3
Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
